# Урок 6. NumPy: от CSV к сравнению пассажиров

**Сегодня:** научимся читать пассажирские данные, создавать массивы NumPy и сравнивать доли выживших в двух возрастных группах.

Готовые демонстрационные ячейки запускайте по порядку. В практиках и ДЗ пишите код в подготовленных ячейках; `? ВЫВОДЫ` — место для вашего объяснения результата. Не нужно добавлять новые рабочие ячейки.

**Для запуска:** этот ноутбук и `titanic.csv` рядом с ним. В Colab загрузите CSV в панель «Файлы». Локально используйте настроенное окружение курса. Подготовка файла ниже — служебная ячейка; её код разбирать на занятии не требуется.

## Знакомимся с данными: пассажиры «Титаника»

«Титаник» вышел из Саутгемптона в Нью-Йорк 10 апреля 1912 года. После столкновения с айсбергом корабль затонул 15 апреля. [Историческая справка: Royal Museums Greenwich](https://www.rmg.co.uk/collections/research-guides/research-guide-d1-rms-titanic-fact-sheet).

Мы работаем с учебной таблицей из **891 записи о пассажирах**, а не с полным списком всех людей на борту. Одна строка — запись об одном пассажире. В ней есть характеристики пассажира и признак `Survived`: 1 — выжил, 0 — не выжил.

**Зачем этот набор в курсе?** На небольшом файле можно пройти весь путь анализа: прочитать таблицу, понять поля, найти пропуски, выбрать группы, сравнить показатели, сохранить результат и позднее построить модель. Сегодня исследуем, как различаются доли выживших в выбранных группах. Наблюдаемое различие не объясняет само по себе причины катастрофы или исход для каждого человека.

**Почему встречаются пустые поля?** Пустое поле означает: значение не указано в этой учебной таблице. По CSV нельзя установить причину каждого пропуска. Значение могли не записать или не включить в набор — это возможные объяснения, а не установленная история конкретного пассажира. Неизвестный возраст не равен нулю; неизвестная каюта не означает, что пассажир ехал без каюты. Мы сохраняем эту неопределённость и учитываем её при расчётах.

Источник таблицы — [учебный набор pandas](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html). Его описание полей и примеры помогут сверить смысл данных.


In [1]:
#@title Подготовка данных — запустите один раз
from pathlib import Path
import hashlib

# Сначала используем выданный файл. Интернет нужен только если файла нет.
DATA_NAME = "titanic.csv"
DATA_SHA256 = "4a437fde05fe5264e1701a7387ac6fb75393772ba38bb2c9c566405af5af4bd7"
candidates = [Path(DATA_NAME), Path("data") / DATA_NAME]
for base in [Path.cwd(), *Path.cwd().parents][:5]:
    candidates.extend([
        base / "07_Данные" / "lesson-06" / DATA_NAME,
        base / "01_Course" / "Python для ИИ" / "07_Данные" / "lesson-06" / DATA_NAME,
    ])
DATA_PATH = next((p for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    from urllib.request import urlopen
    try:
        raw = urlopen("https://raw.githubusercontent.com/pandas-dev/pandas/v2.3.3/doc/data/titanic.csv", timeout=30).read()
    except OSError as error:
        raise FileNotFoundError("Положите выданный titanic.csv рядом с ноутбуком. В Colab загрузите его в панель «Файлы».") from error
    if hashlib.sha256(raw).hexdigest() != DATA_SHA256:
        raise ValueError("Источник данных изменился. Используйте файл, выданный преподавателем.")
    DATA_PATH = Path(DATA_NAME)
    # Режим xb не позволяет случайно перезаписать существующий файл.
    with DATA_PATH.open("xb") as file:
        file.write(raw)
if hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() != DATA_SHA256:
    raise ValueError("Это другая версия titanic.csv. Сохраните её отдельно и используйте выданный учебный файл.")
print("Данные готовы:", DATA_PATH.name)


Данные готовы: titanic.csv


**Подготовка завершена:** дальше читаем содержимое файла. Проверка версии CSV нужна, чтобы у группы были одинаковые исходные данные; она не заменяет проверку записей.

## 🟨❗ Теория. От списка к массиву

Python-список хранит последовательность объектов; NumPy-массив выполняет
операции над всеми элементами сразу. В простом числовом массиве один `dtype`.
`shape` — размеры, `ndim` — число измерений; у одномерного массива форма `(n,)`.
`np.array([1, 2, 3])` создаёт массив. Для умножения используем `*`, без цикла.
Сравните поведение `values * 2` для списка и массива.


In [2]:
import numpy as np
values = [3, 5, 8]
arr = np.array(values)
print("Список * 2:", values * 2)
print("Массив * 2:", arr * 2)
print("Форма / размерность / тип:", arr.shape, arr.ndim, arr.dtype)


Список * 2: [3, 5, 8, 3, 5, 8]
Массив * 2: [ 6 10 16]
Форма / размерность / тип: (3,) 1 int64


## ⭐ Практика 1. Создать массив и выполнить операции

Создайте `numbers = np.array([2, 4, 6, 8])`. Выведите форму shape, тип элементов dtype, удвоенные значения, сумму и среднее исходных чисел. Подпишите результаты. Исходный numbers не изменяйте.

**Зачем:** увидеть, что операция над массивом выполняется для каждого элемента, а сумма и среднее дают разные показатели.

In [ ]:
# Создайте numbers и выведите результаты.


### ❓ Вопросы и выводы

Почему массив * 2 и список * 2 дают разные результаты? Подтвердите ответ числами.


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🟨❗ Теория. Индексы и маска

`arr[0]` — первый элемент; `arr[-1]` — последний; `arr[1:3]` — элементы с индексами 1 и 2.
`arr > 4` создаёт массив True/False. `arr[arr > 4]` выбирает значения по маске.
Несколько условий: `(arr >= 4) & (arr < 8)`; обязательны скобки, для массивов используем `&`, а не `and`.
Маска должна соответствовать массиву по длине и порядку записей.


In [3]:
demo = np.array([10, 20, 30, 40, 50])
print(demo[0], demo[-1], demo[1:4])
mask = (demo >= 20) & (demo < 50)
print("Маска:", mask)
print("Выборка:", demo[mask])


10 50 [20 30 40]
Маска: [False  True  True  True False]
Выборка: [20 30 40]


## ⭐ Практика 2. Проверить границу отбора

Используйте массив numbers со значениями [2, 4, 6, 8] из практики 1. Выведите последние два элемента. Затем выберите значения **от 4 включительно до 8 не включительно**: `(numbers >= 4) & (numbers < 8)`; выведите результат и число элементов. После этого замените `< 8` на `<= 8` и выведите второй результат.

**Зачем:** сравнить два условия на одном массиве и увидеть, как включение границы меняет отбор.

In [ ]:
# Срез, маска и размер выборки.


### ❓ Вопросы и выводы

Попадает ли число 8 из массива numbers в отбор `>= 4` и `< 8`? Что изменилось, когда вы заменили `< 8` на `<= 8`? Назовите оба полученных массива.

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🟨❗ Шаг 1. Прочитать CSV без преобразования значений

Сначала посмотрим, что действительно хранится в файле. `csv.DictReader` читает каждую строку в словарь: ключ — название столбца, значение — текст из ячейки. Даже значение, похожее на число, пока имеет тип `str`. Пустая ячейка читается как пустая строка `""`.

Запустите готовую ячейку. Она только читает записи и показывает поля первой записи, затем начало и конец списка; массивов NumPy здесь ещё нет.

In [4]:
import csv
with DATA_PATH.open(encoding="utf-8", newline="") as file:
    records = list(csv.DictReader(file))
print("Прочитано записей:", len(records))
print("Поля и значения первой записи:")
for field, value in records[0].items():
    print(field, "=", repr(value))
id_text = records[0]["PassengerId"]
print("Тип PassengerId до преобразования:", type(id_text))
print("Первые три записи: ID, исходный Age, Survived")
for row in records[:3]:
    print(row["PassengerId"], repr(row["Age"]), row["Survived"])
print("Последние три записи: те же поля")
for row in records[-3:]:
    print(row["PassengerId"], repr(row["Age"]), row["Survived"])


Прочитано записей: 891
Поля и значения первой записи:
PassengerId = '1'
Survived = '0'
Pclass = '3'
Name = 'Braund, Mr. Owen Harris'
Sex = 'male'
Age = '22'
SibSp = '1'
Parch = '0'
Ticket = 'A/5 21171'
Fare = '7.25'
Cabin = ''
Embarked = 'S'
Тип PassengerId до преобразования: <class 'str'>
Первые три записи: ID, исходный Age, Survived
1 '22' 0
2 '38' 1
3 '26' 1
Последние три записи: те же поля
889 '' 0
890 '26' 1
891 '32' 0


### Что означают поля, с которыми работаем сегодня

| Поле в CSV | По-русски | Пример и смысл |
|---|---|---|
| PassengerId | Идентификатор пассажира | Число связывает запись с исходной таблицей; это не индекс массива |
| Survived | Признак выживания | 0 или 1; сумму можно использовать для подсчёта выживших |
| Pclass | Класс билета | 1, 2 или 3; число обозначает категорию |
| Name, Sex | Имя и пол из таблицы | Текстовые значения; Sex записан как female/male |
| Age | Возраст в годах | Может быть дробным или отсутствовать |
| Fare | Плата за проезд, указанная в записи | Числовой показатель стоимости, а не номер билета |

Номер билета хранится в отдельном поле `Ticket`; каюта — в `Cabin`, порт посадки — в `Embarked`. Полный словарь 12 полей находится в `README_Titanic.md`.

**Как читать Fare:** это записанная в наборе плата за проезд. В самом CSV нет указания денежной единицы и правил распределения стоимости общего билета между пассажирами. Поэтому в заданиях называем показатель Fare, сравниваем его значения и не переводим их в современные цены. Нулевое значение само по себе не доказывает ошибку.


## 🟨❗ Шаг 2. Преобразовать целые числа

Чтобы считать и сравнивать числа, преобразуем их текстовое представление. Сначала покажем **одно значение**: `int("1")` даёт целое число 1. Затем применим тот же шаг к каждой записи и создадим массивы.

PassengerId нужен для связи с записью, Survived — для подсчёта исходов, Pclass — для отбора по классу билета. Числовой тип этих полей не делает их смысл одинаковым.

In [5]:
id_text = records[0]["PassengerId"]
id_number = int(id_text)
print("До:", repr(id_text), type(id_text))
print("После:", id_number, type(id_number))

До: '1' <class 'str'>
После: 1 <class 'int'>


In [6]:
passenger_ids = []
survival_values = []
class_values = []
for row in records:
    passenger_ids.append(int(row["PassengerId"]))
    survival_values.append(int(row["Survived"]))
    class_values.append(int(row["Pclass"]))
passenger_id = np.array(passenger_ids)
survived = np.array(survival_values)
pclass = np.array(class_values)
print("Первые ID:", passenger_id[:5])
print("Первые признаки выживания:", survived[:5])
print("Первые классы билетов:", pclass[:5])
print("Тип массива Survived:", survived.dtype)

Первые ID: [1 2 3 4 5]
Первые признаки выживания: [0 1 1 1 0]
Первые классы билетов: [3 1 3 1 3]
Тип массива Survived: int64


## 🟨❗ Шаг 3. Текст оставить текстом

Имя и категория Sex не нужны для арифметики. Их сохраняем как строки, без `int()` или `float()`. NumPy может хранить массив строк; `dtype` такого массива отличается от числового. Все массивы строим из records в одном порядке: индекс 0 в каждом массиве относится к одной и той же записи.

В выводе `<U6` буква U обозначает строки Unicode, число 6 — вместимость строки до шести символов. Это характеристика хранения массива, а не число пассажиров.

In [7]:
names = np.array([row["Name"] for row in records])
sex = np.array([row["Sex"] for row in records])
print("Первое имя:", names[0])
print("Первые значения Sex:", sex[:5])
print("Тип строкового массива:", sex.dtype)

Первое имя: Braund, Mr. Owen Harris
Первые значения Sex: ['male' 'female' 'female' 'female' 'male']
Тип строкового массива: <U6


## 🟨❗ Шаг 4. Преобразовать дробные числа

Возраст может содержать дробную часть: например, `0.5` года — полгода. Fare тоже может быть дробным, поэтому используем `float()`, а не `int()`.

У Fare в выданной таблице нет пустых ячеек. Сначала покажем одно преобразование, затем весь массив. Перед обработкой другого файла отсутствие пропусков надо проверить заново.

In [8]:
fare_text = records[0]["Fare"]
print("До:", repr(fare_text), type(fare_text))
print("После:", float(fare_text), type(float(fare_text)))
fare = np.array([float(row["Fare"]) for row in records])
print("Первые значения Fare:", fare[:5])

До: '7.25' <class 'str'>
После: 7.25 <class 'float'>
Первые значения Fare: [ 7.25   71.2833  7.925  53.1     8.05  ]


## 🟨❗ Шаг 5. Пустой возраст обозначить как неизвестный

`float("")` не может преобразовать пустую строку в число. Поэтому разделяем два случая: непустой Age преобразуем через `float()`, пустой обозначаем `np.nan`. NaN — специальное обозначение отсутствующего числового значения; это не возраст пассажира и не ноль.

В шестой записи Age пуст: сначала увидим исходный текст, затем построим массив по этому правилу.

In [9]:
print("Age в шестой записи:", repr(records[5]["Age"]))
age_values = []
for row in records:
    age_text = row["Age"]
    if age_text == "":
        age_values.append(np.nan)
    else:
        age_values.append(float(age_text))
age = np.array(age_values)
print("Первые шесть возрастов:", age[:6])
print("Размеры связанных массивов:")
print(passenger_id.size, survived.size, pclass.size, age.size, fare.size, sex.size)

Age в шестой записи: ''
Первые шесть возрастов: [22. 38. 26. 35. 35. nan]
Размеры связанных массивов:
891 891 891 891 891 891


### Паспорт массивов перед исследованием

В NumPy рассматриваем массивы через `shape`, `size` и `dtype`, а начало и конец — через срезы `[:5]` и `[-5:]`. `shape` — форма массива, `size` — число элементов, `dtype` — общий тип его элементов. У этих атрибутов нет скобок. Методы `head()`, `tail()` и `info()` относятся к Pandas и появятся на следующем уроке.

`np.unique(values)` показывает различные значения массива — здесь категории Pclass и Sex. Это сводка категорий, а не список пассажиров. Исходный массив не меняется.

**Зачем:** проверить, что массивы имеют ожидаемую длину и пригодные для дальнейших расчётов типы. Равенство размеров — необходимая проверка, но само по себе не доказывает соответствие записей: поэтому все массивы строим из одного records и в одном порядке.

**Вопрос:** почему у Pclass числовой тип, хотя это категория? Доказывают ли первые пять известных возрастов, что во всём массиве нет NaN? Полный подсчёт пропусков сделаем в шаге про NaN ниже.


In [10]:
for label, values in [("PassengerId", passenger_id), ("Survived", survived),
                      ("Pclass", pclass), ("Age", age), ("Fare", fare), ("Sex", sex)]:
    print(f"{label}: shape={values.shape}, size={values.size}, dtype={values.dtype}")
print("Первые пять возрастов:", age[:5])
print("Последние пять возрастов:", age[-5:])
print("Встречающиеся категории Pclass:", np.unique(pclass))
print("Встречающиеся категории Sex:", np.unique(sex))


PassengerId: shape=(891,), size=891, dtype=int64
Survived: shape=(891,), size=891, dtype=int64
Pclass: shape=(891,), size=891, dtype=int64
Age: shape=(891,), size=891, dtype=float64
Fare: shape=(891,), size=891, dtype=float64
Sex: shape=(891,), size=891, dtype=<U6
Первые пять возрастов: [22. 38. 26. 35. 35.]
Последние пять возрастов: [27. 19. nan 26. 32.]
Встречающиеся категории Pclass: [1 2 3]
Встречающиеся категории Sex: ['female' 'male']


## 🟨❗ Как из 0 и 1 получить число людей и долю

Если Survived равен 1 у двух пассажиров из четырёх, сумма признаков равна 2, а доля — `2 / 4 = 0.5`, то есть 50%. Для массива без пропусков из значений 0/1 `.mean()` выполняет этот же расчёт: сумма делится на число элементов.

Число выживших и доля отвечают на разные вопросы. При сравнении групп рядом с процентом всегда пишем, сколько пассажиров в группе.

In [11]:
example_survived = np.array([1, 0, 1, 0])
print("Выживших:", example_survived.sum())
print("Пассажиров:", example_survived.size)
print("Доля:", example_survived.sum() / example_survived.size)
print(f"То же среднее в процентах: {example_survived.mean():.0%}")

Выживших: 2
Пассажиров: 4
Доля: 0.5
То же среднее в процентах: 50%


## ⭐ Практика 3. Сколько пассажиров выжило в нашей таблице?

По массиву survived выведите число пассажиров, число выживших и долю выживших в процентах. Отдельно выведите минимум, максимум и среднее **показателя платы за проезд Fare** из массива fare. Подпишите каждый результат; используйте методы массива, без цикла по пассажирам.

**Зачем:** проверить объём загруженных данных и получить первые показатели для всей учебной выборки.

In [ ]:
# Рассчитайте показатели survived и fare.


### ❓ Вопросы и выводы

Объясните полученную долю словами: сколько человек выжило из какого числа записей? Почему это доля в нашей учебной таблице, а не автоматически среди всех людей на «Титанике»?

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🟨❗ Почему нельзя сразу усреднить весь Age

В массиве age часть значений — `np.nan`. В NumPy обычное `.mean()` такого массива возвращает NaN: средний возраст без правила обработки пропусков не определён этим расчётом.

`np.isnan(age)` создаёт маску: True там, где возраст отсутствует. `~` меняет True на False и обратно. Поэтому `age[~np.isnan(age)]` выбирает только записанные возраста. Мы исключаем пропуски **из этого расчёта**, а не удаляем пассажиров из исходной таблицы. Покажем правило на трёх значениях перед работой с Titanic.

In [12]:
example_age = np.array([12.0, np.nan, 30.0])
missing_mask = np.isnan(example_age)
known_mask = ~missing_mask
print("Исходные значения:", example_age)
print("Возраст отсутствует:", missing_mask)
print("Возраст указан:", known_mask)
print("Выбранные возраста:", example_age[known_mask])
print("Среднее по всему массиву:", example_age.mean())
print("Среднее по указанным возрастам:", example_age[known_mask].mean())

Исходные значения: [12. nan 30.]
Возраст отсутствует: [False  True False]
Возраст указан: [ True False  True]
Выбранные возраста: [12. 30.]
Среднее по всему массиву: nan
Среднее по указанным возрастам: 21.0


## ⭐ Практика 4. Средний возраст: кого учитываем?

Создайте маску отсутствующего возраста `missing_age_mask = np.isnan(age)` и массив `known_age` только из значений, для которых возраст указан. Выведите число записей с указанным возрастом, число пропусков и средний возраст по known_age в годах. Исходный age не изменяйте, пропуски нулём не заполняйте.

**Зачем:** явно назвать, какие записи участвуют в среднем и какие не участвуют.

In [ ]:
# Создайте маску пропусков и массив указанных возрастов.
# Выведите оба количества и средний возраст с подписью.

### ❓ Вопросы и выводы

Для скольких пассажиров вы рассчитали средний возраст? Почему нельзя разделить сумму указанных возрастов на число всех записей или подставить 0 вместо неизвестного возраста?

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## ⭐ Практика 5. Сравнить пассажиров младше 18 лет и от 18 лет

Сначала запишите предположение: у какой из этих двух групп доля выживших выше? Создайте маски `age < 18` и `age >= 18`. Для каждой группы выведите число пассажиров, число выживших и долю в процентах. Отдельно выведите, сколько записей не попало ни в одну группу из-за пропущенного возраста.

**Зачем:** проверить предположение на данных и сравнить доли с правильным числом людей в каждой группе.

In [ ]:
# До запуска: моё предположение — ...
# Создайте две маски и сравните группы.


### ❓ Вопросы и выводы

Сопоставьте доли двух групп. Подтвердилось ли ваше предположение? Укажите размеры групп и объясните, кого этот вывод не охватывает.

? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## ⭐ Практика 6. Дополнительно: взрослые первого класса

Если осталось время, выберите пассажиров с `Pclass == 1` и `Age >= 18`. Выведите число людей в этой группе и долю выживших. Сравните с долей среди всех пассажиров первого класса; для обоих результатов назовите число записей.

**Зачем:** увидеть, как дополнительное условие меняет группу. Это расширение не заменяет обязательные практики 1–5.

In [ ]:
# Две маски: первый класс и взрослые первого класса.


### ❓ Вопросы и выводы

Почему у двух долей разные знаменатели? Сформулируйте один вывод и один вопрос для следующего урока.


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

# 🏠 Домашнее задание. Продолжить исследование пассажиров

Выполните H1–H3; H4 — дополнительное задание. Используйте массивы, подготовленные выше, и отдельные маски для каждой группы. В каждом обязательном шаге нужны расчёт с подписями и заполненный `? ВЫВОДЫ`. Ориентир: 35–50 минут. Исходный CSV и массивы не меняем.

## 🏠 Домашнее задание H1. Граница возраста: 16 и 18 лет

Отдельно сравните группы age < 16 / age >= 16 и age < 18 / age >= 18. Для каждой группы выведите n, число выживших и долю. Записи без указанного возраста исключены из обеих пар. Можно использовать цикл по порогам [16, 18].


**Зачем этот шаг:** Смена порога проверяет, насколько вывод зависит от определения «ребёнок». Для каждого порога заново посчитайте размеры групп: прежние знаменатели переносить нельзя.

In [ ]:
# Сравнение для двух порогов.


### ❓ Вопросы и выводы

Меняется ли направление различия при смене порога? Подтвердите ответ долями и n; возраст ровно на пороге относится к какой группе?


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание H2. Класс билета

Сравните классы 1 и 3 по численности, доле выживших, среднему значению платы за проезд Fare. Напечатайте показатели с подписями. Не объединяйте наблюдения разного класса.


**Зачем этот шаг:** Доли сравнивают исходы в группах разного размера; среднее значение Fare (плата за проезд) характеризует стоимость внутри каждой группы. Число выживших само по себе недостаточно для сравнения долей выживших.

In [ ]:
# Отберите каждый класс собственной маской.


### ❓ Вопросы и выводы

Совпадает ли направление различия среднего Fare и доли выживших? Почему по этим двум средним нельзя доказать причинное влияние цены?


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание H3. Найти контрпример

Выведите первые пять PassengerId пассажиров первого класса, которые не выжили, и первые пять PassengerId выживших пассажиров третьего класса. Для обеих групп также выведите полное количество.


**Зачем этот шаг:** Контрпример проверяет категоричное утверждение «всегда». Первые пять строк показывают примеры, а полное количество — размер группы; ни один пример не заменяет групповую долю.

In [ ]:
# Два условия на каждый контрпример.


### ❓ Вопросы и выводы

Опровергают ли примеры фразу «первый класс гарантирует выживание»? Опровергают ли они различие групповых долей?


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание H4. Дополнительно: форма массива

Создайте таблицу из двух столбцов Fare и Survived через np.column_stack([fare, survived]). Выведите shape, ndim и первые три строки. Объясните, почему dtype будет общим для обоих столбцов.


**Зачем этот шаг:** Форма проверяет, что получилась таблица с двумя столбцами. Общий тип показывает, как объединение целых и дробных значений меняет представление данных; названия и смысл столбцов надо хранить отдельно.

In [ ]:
# Необязательно: двумерный массив.


### ❓ Вопросы и выводы

Что теряется в числовом массиве по сравнению с таблицей с названиями столбцов?


? ВЫВОДЫ

Ответьте на вопрос выше. Если вы рассчитали показатель, назовите группу, число записей и результат; затем объясните, что из него следует.

## 🏠 Домашнее задание. Проверка перед сдачей

1. Выполните все обязательные домашние шаги и заполните их `? ВЫВОДЫ`.
2. Нажмите Restart Kernel / Restart Runtime, затем Run All. Исправьте ошибки,
   убедитесь, что результаты соответствуют текущему коду, сохраните ноутбук с выводами.
3. В готовом решении не оставляйте обязательные Code-ячейки только с комментариями.
4. Загрузите файл по пути `homework/lesson-06/solution.ipynb` и проверьте его отображение в GitHub.
5. Срок преподаватель объявит отдельно. Новая дата в этом шаблоне не назначена.

**Зачем перезапускать и сохранять:** чистый запуск выявляет зависимость от старых переменных и случайного порядка выполнения; сохранённые выводы позволяют преподавателю сверить ответ с текущим кодом. Просмотр файла в GitHub проверяет, что загружена нужная версия и результаты видны.

**Что проверяется:** работающий код, верные фильтры и знаменатели, сохранённые результаты,
выводы по расчётам, объяснение ограничения. Дополнительное задание не заменяет обязательную часть.

**Следующий урок:** те же данные и те же вопросы в Pandas.

## Источники
- [Набор Titanic в документации pandas](https://raw.githubusercontent.com/pandas-dev/pandas/v2.3.3/doc/data/titanic.csv)
- [NumPy: основы](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Pandas: учебные руководства](https://pandas.pydata.org/docs/getting_started/intro_tutorials/)


**Сквозная работа:** сохраните исходный `titanic.csv`; на уроке 7 будете использовать тот же файл, сверите расчёты и сохраните личную проверенную таблицу для урока 8. Паспорт и маршрут — в `README_Titanic.md`.

## Справка. Проверки данных перед анализом

Проверка отвечает на вопрос «можно ли доверять этому расчёту?». Очистка — обоснованное изменение данных после найденной проблемы. Один необычный результат ещё не объясняет, что исправлять.

| Что проверяем | Какую проблему ищем | Что делать при проблеме |
|---|---|---|
| Файл, число записей, одинаковые длины и порядок массивов | Другой набор или несовпадение пассажиров между столбцами | Проверить загрузку. Не фильтровать связанные массивы независимо |
| `dtype` и смысл числового поля | Строки вместо чисел, неверное преобразование | Проверить исходное значение и способ чтения; не подменять ошибку нулём |
| `np.isnan(age)` и число известных возрастов | Неполная информация для возрастного вопроса | Для этого расчёта выбрать известные возраста и сообщить число исключённых |
| Число строк после маски, границы < / >= | Неверная группа или потерянная граница | Исправить условие; делить на размер выбранной группы |
| Минимум/максимум Fare и подозрительная исходная запись | Возможная ошибка стоимости | Сверить запись с источником: высокая цена и нулевая цена сами по себе не доказывают ошибку |

**До интерпретации:** `Survived` должен означать 0/1, `Pclass` — 1/2/3; известные Age и Fare не должны быть отрицательными. Если значения нарушают смысл поля, сначала выясняем причину, затем считаем показатель. Для массива 0/1 среднее — доля только при корректной кодировке.

**Почему не удаляем все строки с пропусками?** Для вопроса о возрасте нужен известный Age; для сравнения классов неизвестный возраст не мешает. Маска создаёт выборку для конкретного вопроса, сохраняя исходные наблюдения. Пропуски других полей не повод исключать пассажира из любого расчёта.
